# NHL Data Pipeline — Games

In [ ]:
import requests
import sqlite3
import pandas as pd
import time

DB_PATH = "../data/nhl.db"
# Pinned to a completed season — "now" resolves to the upcoming season
# during the off-season, which has no played games yet.
SEASON = "20252026"
SCHEDULE_URL = "https://api-web.nhle.com/v1/club-schedule-season/{team}/" + SEASON
TIMEOUT = 15
SLEEP = 0.4

conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()

teams = pd.read_sql("SELECT team_id, team_abbrev FROM teams ORDER BY team_abbrev", conn)
team_id_lookup = dict(zip(teams.team_abbrev, teams.team_id))

In [ ]:
def parse_schedule(schedule, team_id_lookup):
    games = []
    for g in schedule.get("games", []):
        home, away = g["homeTeam"]["abbrev"], g["awayTeam"]["abbrev"]
        if home not in team_id_lookup or away not in team_id_lookup:
            continue
        games.append({
            "game_id": g["id"],
            "season": str(g.get("season")),
            "game_type": g.get("gameType"),
            "game_date": g.get("gameDate"),
            "home_team_id": team_id_lookup[home],
            "away_team_id": team_id_lookup[away],
            "home_score": g["homeTeam"].get("score"),
            "away_score": g["awayTeam"].get("score"),
            "game_state": g.get("gameState"),
            "venue_name": g.get("venue", {}).get("default"),
        })
    return games

In [ ]:
all_games, failed_teams = [], []

for row in teams.itertuples(index=False):
    try:
        resp = requests.get(SCHEDULE_URL.format(team=row.team_abbrev), timeout=TIMEOUT)
        resp.raise_for_status()
        all_games.extend(parse_schedule(resp.json(), team_id_lookup))
    except Exception as e:
        failed_teams.append((row.team_abbrev, str(e)))
    time.sleep(SLEEP)

# each game appears in both teams' schedules
games = pd.DataFrame(all_games).drop_duplicates(subset="game_id")
print(f"{len(games)} unique games, {len(failed_teams)} team requests failed")

In [ ]:
for row in games.itertuples(index=False):
    cur.execute(
        "INSERT OR IGNORE INTO games (game_id, season, game_type, game_date, home_team_id, "
        "away_team_id, home_score, away_score, game_state, venue_name) "
        "VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)",
        row,
    )
conn.commit()

# refresh scores/state for games that were already loaded as FUT
for row in games.itertuples(index=False):
    cur.execute(
        "UPDATE games SET home_score = ?, away_score = ?, game_state = ? "
        "WHERE game_id = ? AND (game_state != ? OR game_state IS NULL)",
        (row.home_score, row.away_score, row.game_state, row.game_id, row.game_state),
    )
conn.commit()
conn.close()